# 04_DIM_WEATHER

## Objective
Create the Weather Dimension for the Data Warehouse.

## Notes
- Surrogate key `weatherSK` is generated with Delta Identity.
- Three rows only: 'Rain', 'No Rain', 'Weather Unknown'.

# Setup

In [1]:
from pyspark.sql import SparkSession
from delta.tables import DeltaTable, IdentityGenerator
from pyspark.sql.types import LongType, StringType, BooleanType
from pyspark.sql.functions import *
import ConnectionConfig as cc

# Debug mode
debugging_mode = True

# Initialize Spark


In [2]:
cc.setupEnvironment()
spark = cc.startLocalCluster("DIM_WEATHER")
cc.set_connectionProfile("catchem")
spark.getActiveSession()

Running in Docker environment.
Setting JAVA_HOME to /usr/lib/jvm/java-17-openjdk-arm64
Running in Docker: using pre-installed Spark + JARs


#  Load dim_season from CSV


In [9]:
# EXTRACT
weather_csv_path = "data/Extra_Data/dim_weather.csv"

df_weather = spark.read.option("header", True) \
                       .option("inferSchema", True) \
                       .csv(weather_csv_path)


In [10]:
if debugging_mode:
    df_weather.show()

+----------+---------------+
|weather_id|           name|
+----------+---------------+
|         1|           Rain|
|         2|        No Rain|
|         3|Weather Unknown|
+----------+---------------+



# TRANSFORM
- Only keep necessary columns (name)

In [4]:
dim_weather_transformed = df_weather.select("name")


In [ ]:
if debugging_mode:
    dim_weather_transformed.show()

# LOAD — Create Delta table with Identity

In [ ]:
# Drop existing table if it exists for clean run
if DeltaTable.isDeltaTable(spark, "dim_weather"):
    DeltaTable.forName(spark, "dim_weather").drop()

In [5]:
cc.delete_delta_table(spark, "dim_weather")

Successfully deleted metadata and files for dim_weather


In [6]:
# LOAD

# Create new table with identity column starting at 1
DeltaTable.create(spark) \
    .tableName("dim_weather") \
    .addColumn("weather_sk", LongType(), nullable=False, generatedAlwaysAs=IdentityGenerator(1,1)) \
    .addColumn("name", StringType()) \
    .property("delta.feature.identityColumns", "supported") \
    .execute()

# Insert data into Delta table

In [7]:
# LOAD (Write data)

dim_weather_transformed.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("dim_weather")



In [ ]:
# Delete parquet file if it already exists
cc.delete_parquet_table("parquet/dim_weather_parquet")

In [ ]:
# Save as Parquet format

df = spark.read.format("delta").load("spark-warehouse/dim_weather")

df.coalesce(1).write.format("parquet").save("./parquet/dim_weather_parquet")

In [8]:
# Verify
spark.read.format("delta").load("spark-warehouse/dim_weather").show()

+----------+---------------+
|weather_sk|           name|
+----------+---------------+
|         1|           Rain|
|         2|        No Rain|
|         3|Weather Unknown|
+----------+---------------+



In [1]:
spark.stop()

NameError: name 'spark' is not defined

In [12]:
%reset